# NeSy-SMP 재현 (Colab + Google Drive) — Strategy C

로컬/`/content`에 86GB DB를 **복사하지 않습니다**. Drive만 마운트하고, 쿼리 결과는 작은 CSV로 `NeSy-SMP-derived/`에만 저장합니다.

**데이터:** [Datasets-new](https://drive.google.com/drive/folders/1xOrVTojve-svw0QuHYu0aT-vSJORCLfY)  
**이 노트북/코드:** [NeSy-SMP-repro](https://github.com/dlwldn4824/NeSy-SMP-repro)  
**Upstream:** [FabrizioDeSantis/NeSy-SMP](https://github.com/FabrizioDeSantis/NeSy-SMP)

| 하지 말 것 | 할 것 |
|---|---|
| ECG zip 35GB 복사 | Drive에서 DB **읽기 전용** 쿼리 |
| `MIMIC4-hosp-icu.db`를 `/content`로 cp | 산출물 → `MyDrive/NeSy-SMP-derived/*.csv` |
| 세션마다 DB 재다운로드 | 코호트 → 이벤트 → 6h CSV 순으로 캐시 |

> 런타임: **런타임 → 런타임 유형 변경 → GPU** (추출만이면 CPU도 OK)  
> **지금 할 일:** 셀 0→8.5까지 실행 → `cohort_sepsis_v1.csv` → n/mortality를 채팅에 붙여넣기  
> 노트북이 GitHub보다 최신이면: 로컬 `C:\dev\NeSy-SMP-repro\NeSy-SMP_Colab.ipynb`를 Colab에 **업로드**해서 여세요.

## 0. Drive 마운트

공유 폴더가 내 Drive에 안 보이면, Drive 웹에서 해당 폴더 → **내 Drive에 바로가기 추가**를 먼저 하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls -lh "/content/drive/MyDrive"

## 1. 데이터 경로 찾기

폴더 이름이 다르면 `DRIVE_DATA`만 수정하세요.

In [ ]:
from pathlib import Path
import os

# 공유 폴더를 "내 Drive에 바로가기"로 추가했다고 가정
# 실제 경로가 다르면 여기를 수정
CANDIDATES = [
    Path('/content/drive/MyDrive/Datasets-new'),
    Path('/content/drive/MyDrive/학연생/Datasets-new'),
    Path('/content/drive/Shareddrives'),  # 공유 드라이브인 경우
]

DRIVE_DATA = None
for p in CANDIDATES:
    if p.exists():
        print('found root:', p)
        # MIMIC db가 들어있는 하위 폴더 탐색
        hits = list(p.rglob('MIMIC4-hosp-icu.db'))
        if hits:
            DRIVE_DATA = hits[0].parent
            break

# 수동 지정 예시:
# DRIVE_DATA = Path('/content/drive/MyDrive/Datasets-new')

assert DRIVE_DATA is not None, 'MIMIC4-hosp-icu.db 경로를 찾지 못했습니다. DRIVE_DATA를 수동 지정하세요.'
DB_PATH = DRIVE_DATA / 'MIMIC4-hosp-icu.db'
print('DRIVE_DATA =', DRIVE_DATA)
print('DB exists =', DB_PATH.exists(), DB_PATH)
!ls -lh "{DRIVE_DATA}"

## 2. 디스크 여유 확인

| 전략 | 조건 | 설명 |
|---|---|---|
| A. Colab 로컬로 복사 | 여유 ≥ 100GB | SQLite가 빠름. Colab Pro 등 |
| B. Drive 경로 직접 쿼리 | 여유 부족 | 가능하지만 **매우 느림** |
| C. 파생 CSV만 Drive에 저장 (권장) | 여유 부족 | 한 번 추출 후 작은 CSV로 학습 |

In [ ]:
import shutil

total, used, free = shutil.disk_usage('/content')
print(f'Colab /content free: {free / 2**30:.1f} GB')

STRATEGY = 'C'  # 'A' | 'B' | 'C'
if free / 2**30 >= 100:
    print('여유 충분 → STRATEGY=A 가능')
else:
    print('여유 부족 → STRATEGY=B 또는 C 권장 (기본 C)')

## 3. 코드 clone + 패키지

In [ ]:
%cd /content
# reproduction workspace (code under NeSy-SMP/)
!git clone https://github.com/dlwldn4824/NeSy-SMP-repro.git
%cd /content/NeSy-SMP-repro/NeSy-SMP

!pip -q install ltn rdflib networkx pyvis xgboost medspacy sqlalchemy seaborn

## 4A. (선택) DB를 Colab 로컬로 복사

여유 디스크가 있을 때만 실행. Drive 위 SQLite보다 훨씬 빠릅니다.

In [ ]:
if STRATEGY == 'A':
    LOCAL_DB = Path('/content/MIMIC4-hosp-icu.db')
    if not LOCAL_DB.exists():
        print('copying DB... (수십 분 걸릴 수 있음)')
        !cp "{DB_PATH}" "{LOCAL_DB}"
    WORK_DB = LOCAL_DB
else:
    WORK_DB = DB_PATH
    print('Drive 경로를 그대로 사용:', WORK_DB)

print('WORK_DB =', WORK_DB)

## 4B/C. DB 테이블 목록 확인

In [ ]:
import sqlite3
import pandas as pd

# Drive 위 SQLite는 timeout을 넉넉히
conn = sqlite3.connect(f'file:{WORK_DB}?mode=ro', uri=True, timeout=60)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print(tables.head(50))
print('num tables:', len(tables))
conn.close()

## 5. (권장) 필요한 테이블만 작은 산출물로 Drive에 저장

86GB 전체를 매번 쓰지 말고, **한 번 추출한 CSV/Parquet**를 Drive에 두고 재사용하세요.

아래는 스키마 확인용 예시입니다. 실제 sepsis 코호트 SQL은 테이블명에 맞게 수정해야 합니다.

In [ ]:
OUT_DIR = Path('/content/drive/MyDrive/NeSy-SMP-derived')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('derived output →', OUT_DIR)

conn = sqlite3.connect(f'file:{WORK_DB}?mode=ro', uri=True, timeout=120)

# 예시: patients / admissions 미리보기 (테이블명이 다르면 수정)
for t in ['patients', 'admissions', 'icustays']:
    try:
        n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0, 0]
        print(f'{t}: {n:,} rows')
    except Exception as e:
        print(f'{t}: ERROR {e}')

conn.close()

## 6. notes zip (동반질환용)

notes는 ~1.83GB라 Colab 로컬로 복사하기 쉽습니다.

In [ ]:
note_zips = list(DRIVE_DATA.glob('*note*.zip')) + list(DRIVE_DATA.glob('*note*'))
print('note candidates:', note_zips)

# 예:
# !cp "{DRIVE_DATA}/mimic-iv-note-deidentified-free-text-clinical-notes-2.zip" /content/
# !unzip -q /content/mimic-iv-note-*.zip -d /content/mimic-iv-note

## 7. 학습용 CSV가 준비된 뒤

전처리로 `events_6h_before_death_gcs.csv`를 만들었으면 Drive 파생 폴더에 두고 symlink/copy 후 학습합니다.

In [ ]:
# 예시: Drive에 이미 만든 CSV를 repo가 기대하는 위치로 연결
# !mkdir -p /content/NeSy-SMP-repro/NeSy-SMP/data/subset
# !cp "{OUT_DIR}/events_6h_before_death_gcs.csv" /content/NeSy-SMP-repro/NeSy-SMP/data/subset/

# %cd /content/NeSy-SMP-repro/NeSy-SMP
# !python main.py

## 팁

1. **ECG zip(35GB)은 받지/복사하지 마세요.** 이 논문에 안 씁니다.
2. Drive 위 SQLite는 느립니다. 가능하면 필요한 쿼리 결과를 `NeSy-SMP-derived/`에 CSV로 캐시하세요.
3. Colab 세션이 끊기면 `/content`는 사라집니다. **산출물은 항상 Drive에 저장**하세요.
4. Free Colab 디스크가 부족하면 Colab Pro 또는 GCP VM을 쓰고, 로컬 Mac에는 안 받는 전략을 유지하세요.
5. 공유 폴더 접근 권한(보기/다운로드)이 있어야 `drive.mount` 후 파일이 보입니다.

## 8. Sepsis 코호트 추출 (이어받기 가능)

Drive 위 SQLite는 느리고 Colab이 끊길 수 있습니다. **한 방에 조인하지 말고** 중간 CSV를 Drive에 남깁니다.

| 단계 | 파일 | 내용 |
|---|---|---|
| 8.1 | `_cache/icu_one.csv` | ICU 1회 + stay ≥ 24h |
| 8.2 | `_cache/sepsis_hadm.csv` | sepsis ICD hadm_id |
| 8.3 | `_cache/patients_min.csv` | age/gender |
| 8.4 | `_cache/admissions_min.csv` | 사망·입원 시각 |
| 8.5 | `cohort_sepsis_v1.csv` | pandas 조인 최종본 |

이미 있는 단계는 **자동 스킵**됩니다. 끊기면 런타임 재연결 → 0~4 셀 다시 실행 → **이 섹션만 재실행**하세요.

논문 sanity: n ≈ **19,328**, mortality ≈ **18%** (ICD-only라 완전 일치 불필요)

In [ ]:
from pathlib import Path
import sqlite3
import time
import pandas as pd

# 위 셀에서 DRIVE_DATA / DB_PATH가 있으면 재사용, 없으면 기본 경로
try:
    DRIVE_DATA
    DB_PATH
except NameError:
    DRIVE_DATA = Path('/content/drive/MyDrive/Datasets-new')
    DB_PATH = DRIVE_DATA / 'MIMIC4-hosp-icu.db'

WORK_DB = DB_PATH  # STRATEGY A면 /content/MIMIC4-hosp-icu.db 로 바꿔도 됨
OUT_DIR = Path('/content/drive/MyDrive/NeSy-SMP-derived')
CACHE = OUT_DIR / '_cache'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

assert WORK_DB.exists(), f'DB 없음: {WORK_DB}'

def connect_ro(timeout=600):
    return sqlite3.connect(f'file:{WORK_DB}?mode=ro', uri=True, timeout=timeout)

def load_or_query(path: Path, sql: str, desc: str) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        df = pd.read_csv(path)
        print(f'[SKIP] {desc}: {path.name} ({len(df):,} rows)')
        return df
    print(f'[RUN ] {desc} ...')
    t0 = time.time()
    conn = connect_ro()
    try:
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()
    df.to_csv(path, index=False)
    print(f'[DONE] {desc}: {len(df):,} rows → {path} ({time.time()-t0:.0f}s)')
    return df

print('WORK_DB =', WORK_DB)
print('OUT_DIR  =', OUT_DIR)
print('CACHE    =', CACHE)
for p in sorted(CACHE.glob('*.csv')):
    print(f'  cached: {p.name} ({p.stat().st_size/1e6:.1f} MB)')
cohort_path = OUT_DIR / 'cohort_sepsis_v1.csv'
print('cohort exists:', cohort_path.exists(), cohort_path)

In [ ]:
# 8.1–8.4: 중간 테이블 추출 (있으면 스킵)

icu_one = load_or_query(
    CACHE / 'icu_one.csv',
    """
    WITH icu_ranked AS (
      SELECT
        subject_id, hadm_id, stay_id, intime, outtime, los,
        COUNT(*) OVER (PARTITION BY hadm_id) AS n_icu
      FROM icustays
    )
    SELECT subject_id, hadm_id, stay_id, intime, outtime, los
    FROM icu_ranked
    WHERE n_icu = 1 AND los >= 1.0
    """,
    'ICU single-stay ≥24h',
)

sepsis_hadm = load_or_query(
    CACHE / 'sepsis_hadm.csv',
    """
    SELECT DISTINCT hadm_id
    FROM diagnoses_icd
    WHERE
      (icd_version = 10 AND (
        icd_code LIKE 'A40%' OR icd_code LIKE 'A41%' OR icd_code LIKE 'R652%'
      ))
      OR (icd_version = 9 AND icd_code IN ('99591', '99592', '78552'))
    """,
    'sepsis ICD hadm_id (느릴 수 있음)',
)

patients_min = load_or_query(
    CACHE / 'patients_min.csv',
    "SELECT subject_id, gender, anchor_age FROM patients",
    'patients (min cols)',
)

admissions_min = load_or_query(
    CACHE / 'admissions_min.csv',
    """
    SELECT hadm_id, subject_id, admittime, dischtime, deathtime,
           admission_location, hospital_expire_flag
    FROM admissions
    """,
    'admissions (min cols)',
)

print('--- cache ready ---')
print('icu_one', len(icu_one), '| sepsis_hadm', len(sepsis_hadm),
      '| patients', len(patients_min), '| admissions', len(admissions_min))

# 8.5: pandas 조인 → 최종 코호트 (DB 재쿼리 없음)

cohort_path = OUT_DIR / 'cohort_sepsis_v1.csv'
if cohort_path.exists() and cohort_path.stat().st_size > 0:
    cohort = pd.read_csv(cohort_path)
    print(f'[SKIP] cohort already exists: {cohort_path} ({len(cohort):,} rows)')
else:
    cohort = (
        icu_one.merge(sepsis_hadm, on='hadm_id', how='inner')
               .merge(patients_min, on='subject_id', how='inner')
               .merge(
                   admissions_min.drop(columns=['subject_id'], errors='ignore'),
                   on='hadm_id', how='inner',
               )
    )
    cohort = cohort[cohort['anchor_age'] >= 18].copy()
    cohort = cohort.rename(columns={
        'anchor_age': 'age',
        'intime': 'icu_intime',
        'outtime': 'icu_outtime',
        'los': 'icu_los_days',
    })
    keep = [
        'subject_id', 'hadm_id', 'stay_id', 'gender', 'age',
        'admittime', 'dischtime', 'deathtime', 'admission_location',
        'hospital_expire_flag', 'icu_intime', 'icu_outtime', 'icu_los_days',
    ]
    cohort = cohort[[c for c in keep if c in cohort.columns]]
    cohort.to_csv(cohort_path, index=False)
    print(f'[DONE] saved {cohort_path}')

n = len(cohort)
mort = float(cohort['hospital_expire_flag'].mean()) if n else float('nan')
print(f'n_patients (rows): {n:,}')
print(f'mortality: {mort:.1%}')
print('paper target: n≈19,328 / mortality≈18%')
print(cohort.head())
print('\n→ 이 n / mortality 숫자를 채팅에 붙여 주세요. 다음: 이벤트 로그 추출.')

### 실행 체크
1. 가장 오래 걸리는 단계는 보통 **8.2 sepsis ICD** (`diagnoses_icd` 스캔).
2. 중간에 끊겨도 `_cache/*.csv`가 Drive에 남아 있으면 그 단계는 스킵됩니다.
3. 캐시를 지우고 다시 뽑으려면 Drive에서 `NeSy-SMP-derived/_cache/` 또는 `cohort_sepsis_v1.csv`를 삭제한 뒤 재실행.
4. 끝나면 `n_patients` / `mortality`를 채팅에 붙여 주세요 → 이벤트(vitals/labs) 추출으로 진행합니다.

## 9. 이벤트 로그 추출 (cohort → vitals/labs CSV)

**코호트 v1 결과:** n≈9,974 / mortality≈26.3%  
(논문 19,328 / 18%는 Sepsis-3+SOFA 기준. ICD-only v1로 파이프라인 먼저 완주.)

`chartevents` / `labevents`는 큼 → **stay_id 배치**로 뽑아 Drive `_cache/`에 이어 저장합니다.  
끊기면 같은 셀 재실행 → 이미 있는 배치 파일은 스킵.

In [ ]:
# 9.0 준비: cohort 로드 + itemid 맵
from pathlib import Path
import sqlite3, time, json
import pandas as pd
import numpy as np

OUT_DIR = Path('/content/drive/MyDrive/NeSy-SMP-derived')
CACHE = OUT_DIR / '_cache'
EVT = CACHE / 'events_batches'
EVT.mkdir(parents=True, exist_ok=True)

cohort = pd.read_csv(OUT_DIR / 'cohort_sepsis_v1.csv')
print('cohort', len(cohort), 'mort', f"{cohort['hospital_expire_flag'].mean():.1%}")

# MIMIC-IV itemids (metavision 위주; 필요시 확장)
CHART_ITEMS = {
    # concept:name -> itemids
    'HR': [220045],
    'SBP': [220050, 220179],
    'DBP': [220051, 220180],
    'MAP': [220052, 220181, 225312],
    'RR': [220210, 224690],
    'TempC': [223762],
    'TempF': [223761],
    'GCS_Eye': [220739],
    'GCS_Verbal': [223900],
    'GCS_Motor': [223901],
}
LAB_ITEMS = {
    'Lactate': [50813],
    'Creatinine': [50912],
    'Hemoglobin': [51222],
    'Platelets': [51265],
    'Bilirubin': [50885],
    'Potassium': [50971],
    'Albumin': [50862],
    'CRP': [50889],
    'Glucose': [50931, 50809],
    'WBC': [51301],
    'ALT': [50861],
    'AST': [50878],
    'Lymphocytes': [51244],
    'Neutrophils': [51256],
}

chart_id_to_name = {i: n for n, ids in CHART_ITEMS.items() for i in ids}
lab_id_to_name = {i: n for n, ids in LAB_ITEMS.items() for i in ids}
all_chart_ids = sorted(chart_id_to_name)
all_lab_ids = sorted(lab_id_to_name)

stay_ids = cohort['stay_id'].astype(int).unique().tolist()
hadm_ids = cohort['hadm_id'].astype(int).unique().tolist()
BATCH = 250
print('stays', len(stay_ids), 'hadms', len(hadm_ids), 'batch', BATCH)

try:
    WORK_DB
except NameError:
    WORK_DB = Path('/content/drive/MyDrive/Datasets-new/MIMIC4-hosp-icu.db')

def connect_ro(timeout=600):
    return sqlite3.connect(f'file:{WORK_DB}?mode=ro', uri=True, timeout=timeout)

print('WORK_DB', WORK_DB)

In [ ]:
# 9.1 chartevents 배치 추출 (이어받기)
chart_ids_csv = ','.join(map(str, all_chart_ids))
n_batches = (len(stay_ids) + BATCH - 1) // BATCH
print(f'chart batches: {n_batches}')

for b in range(n_batches):
    out = EVT / f'chart_batch_{b:04d}.csv'
    if out.exists() and out.stat().st_size > 0:
        print(f'[SKIP] {out.name}')
        continue
    batch = stay_ids[b*BATCH:(b+1)*BATCH]
    ids = ','.join(map(str, batch))
    sql = f"""
      SELECT stay_id, subject_id, hadm_id, itemid, charttime, valuenum
      FROM chartevents
      WHERE stay_id IN ({ids})
        AND itemid IN ({chart_ids_csv})
        AND valuenum IS NOT NULL
    """
    t0 = time.time()
    conn = connect_ro()
    try:
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()
    df.to_csv(out, index=False)
    print(f'[DONE] {out.name}: {len(df):,} rows ({time.time()-t0:.0f}s) [{b+1}/{n_batches}]')

print('chartevents batches complete')

In [ ]:
# 9.2 labevents 배치 추출 (이어받기) — hadm_id 기준
lab_ids_csv = ','.join(map(str, all_lab_ids))
n_batches = (len(hadm_ids) + BATCH - 1) // BATCH
print(f'lab batches: {n_batches}')

for b in range(n_batches):
    out = EVT / f'lab_batch_{b:04d}.csv'
    if out.exists() and out.stat().st_size > 0:
        print(f'[SKIP] {out.name}')
        continue
    batch = hadm_ids[b*BATCH:(b+1)*BATCH]
    ids = ','.join(map(str, batch))
    sql = f"""
      SELECT subject_id, hadm_id, itemid, charttime, valuenum
      FROM labevents
      WHERE hadm_id IN ({ids})
        AND itemid IN ({lab_ids_csv})
        AND valuenum IS NOT NULL
    """
    t0 = time.time()
    conn = connect_ro()
    try:
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()
    df.to_csv(out, index=False)
    print(f'[DONE] {out.name}: {len(df):,} rows ({time.time()-t0:.0f}s) [{b+1}/{n_batches}]')

print('labevents batches complete')

In [ ]:
# 9.3 배치 병합 → event log (dataset_gcs 전단계)
from pathlib import Path

def load_batches(prefix):
    files = sorted(EVT.glob(f'{prefix}_batch_*.csv'))
    if not files:
        raise FileNotFoundError(f'no {prefix} batches in {EVT}')
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

chart = load_batches('chart')
lab = load_batches('lab')
print('raw chart', len(chart), 'lab', len(lab))

chart['concept:name'] = chart['itemid'].map(chart_id_to_name)
lab['concept:name'] = lab['itemid'].map(lab_id_to_name)

# °F → °C
mask_f = chart['concept:name'] == 'TempF'
chart.loc[mask_f, 'valuenum'] = (chart.loc[mask_f, 'valuenum'] - 32) * 5 / 9
chart.loc[mask_f, 'concept:name'] = 'TempC'

# GCS total
gcs_parts = chart[chart['concept:name'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor'])].copy()
if len(gcs_parts):
    gcs_parts['time_bin'] = pd.to_datetime(gcs_parts['charttime']).dt.floor('h')
    gcs = (gcs_parts.pivot_table(
        index=['stay_id', 'subject_id', 'hadm_id', 'time_bin'],
        columns='concept:name', values='valuenum', aggfunc='last'
    ).reset_index())
    need = ['GCS_Eye', 'GCS_Verbal', 'GCS_Motor']
    if all(c in gcs.columns for c in need):
        gcs['valuenum'] = gcs[need].sum(axis=1)
        gcs['concept:name'] = 'GCS'
        gcs['charttime'] = gcs['time_bin']
        gcs = gcs[['stay_id', 'subject_id', 'hadm_id', 'charttime', 'valuenum', 'concept:name']]
    else:
        gcs = pd.DataFrame()
else:
    gcs = pd.DataFrame()

chart = chart[~chart['concept:name'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor', 'TempF'])]

# lab에 stay_id 붙이기 (cohort 기준)
lab = lab.merge(cohort[['hadm_id', 'stay_id']], on='hadm_id', how='inner')

events = pd.concat([
    chart[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'concept:name', 'valuenum']],
    lab[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'concept:name', 'valuenum']],
    gcs[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'concept:name', 'valuenum']] if len(gcs) else gcs,
], ignore_index=True)

events = events.merge(
    cohort[['hadm_id', 'hospital_expire_flag', 'deathtime', 'icu_intime', 'icu_outtime', 'age', 'admission_location']],
    on='hadm_id', how='left'
)
events = events.rename(columns={'charttime': 'time:timestamp', 'valuenum': 'value'})
events['time:timestamp'] = pd.to_datetime(events['time:timestamp'])

# ICU 구간만
events['icu_intime'] = pd.to_datetime(events['icu_intime'])
events['icu_outtime'] = pd.to_datetime(events['icu_outtime'])
events = events[(events['time:timestamp'] >= events['icu_intime']) &
                (events['time:timestamp'] <= events['icu_outtime'])]

# Death 이벤트 (사망자)
dead = cohort[cohort['hospital_expire_flag'] == 1].copy()
dead['deathtime'] = pd.to_datetime(dead['deathtime'])
death_rows = dead.dropna(subset=['deathtime'])[['subject_id', 'hadm_id', 'stay_id', 'deathtime', 'hospital_expire_flag', 'age', 'admission_location']].copy()
death_rows['time:timestamp'] = death_rows['deathtime']
death_rows['concept:name'] = 'Death'
death_rows['value'] = 1.0

events = pd.concat([events, death_rows[events.columns.intersection(death_rows.columns)]], ignore_index=True)
events = events.sort_values(['hadm_id', 'time:timestamp'])

out_events = OUT_DIR / 'dataset_gcs_v1.csv'
events.to_csv(out_events, index=False)
print('saved', out_events)
print('rows', f'{len(events):,}', 'hadm', events['hadm_id'].nunique())
print(events['concept:name'].value_counts().head(20))
print(events.head())
print('\n→ rows / hadm / concept value_counts 상단을 채팅에 붙여 주세요.')